<div style="background-color:#0B0C0F;border-bottom:1px solid #2A2E37;padding:40px 32px;background-image: linear-gradient(rgba(255,255,255,.035) 1px, transparent 1px), linear-gradient(90deg, rgba(255,255,255,.035) 1px, transparent 1px); background-size: 24px 24px; max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 10px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">Observatoire : Marché data France</p>
<h1 style="margin:0;padding:0;font-size:34px;letter-spacing:-0.02em;color:#F2F3F5;">EXPLORATION DU MARCHÉ DE L'EMPLOI DATA EN FRANCE</h1>
</div>

<div style="padding:8px 0 24px 0;max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 8px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">Objectif</p>
<h2 style="margin:0 0 16px 0;padding:0;letter-spacing:-0.02em;color:#F2F3F5;">Objectif de ce notebook</h2>
<p style="color:#F2F3F5;line-height:1.65;">Ce notebook explore <strong>fct_offre</strong> et ses tables satellites, telles que le pipeline dbt les produit. Il ne transforme rien : il interroge, mesure et commente. Toute correction identifiée ici remonte dans les modèles, jamais dans le notebook.</p>
<p style="color:#F2F3F5;line-height:1.65;">Corpus au 30/08/2026 : <strong>960 offres</strong> collectées depuis l'API France Travail sur les codes ROME M1405 et M1811, plus quatre mots clés ciblés. Ces 960 offres correspondent à <strong>865 annonces distinctes</strong>. L'écart n'est pas anecdotique : il renverse deux classements.</p>
<p style="color:#F2F3F5;line-height:1.65;">Le notebook suit sept étapes, groupées en trois temps : décrire le corpus, établir ce qu'il n'est <em>pas</em> (trois limites de comptage mesurables, chacune faussant une conclusion différente), puis analyser le marché en tenant compte de ces limites.</p>
</div>

<div style="background-color:#15171C;border-bottom:1px solid #2A2E37;padding:28px 32px;background-image: linear-gradient(rgba(255,255,255,.035) 1px, transparent 1px), linear-gradient(90deg, rgba(255,255,255,.035) 1px, transparent 1px); background-size: 24px 24px; max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 8px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">01. ÉTAPE</p>
<h2 style="margin:0;padding:0;letter-spacing:-0.02em;color:#F2F3F5;">Le corpus, tel qu'il est</h2>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">1.1</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Connexion et périmètre</h3>
</div>

In [1]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

pd.set_option("display.max_columns", None)

# Lecture seule : le notebook n'a rien à écrire dans le warehouse, et
# DuckDB étant mono-writer, la connexion en lecture seule évite aussi
# qu'un notebook resté ouvert bloque un run dbt en parallèle.
con = duckdb.connect("../data/warehouse.duckdb", read_only=True)

In [2]:
# Thème appliqué à toutes les figures du notebook.
pio.templates["millimeter"] = go.layout.Template(
    layout=go.Layout(
        paper_bgcolor="#15171C",
        plot_bgcolor="#15171C",
        font=dict(family="'JetBrains Mono', monospace", color="#9BA1AC", size=11),
        title=dict(font=dict(family="Inter, sans-serif", color="#F2F3F5", size=15), x=0),
        colorway=["#4C8DFF", "#FFC24B", "#FF6B5A", "#4ADE80"],
        xaxis=dict(gridcolor="#2A2E37", zerolinecolor="#2A2E37", linecolor="#2A2E37"),
        yaxis=dict(gridcolor="#2A2E37", zerolinecolor="#2A2E37", linecolor="#2A2E37"),
        margin=dict(l=20, r=40, t=50, b=40),
    )
)
pio.templates.default = "millimeter"

In [3]:
con.execute("""
    select
        count(*) as offres,
        count(case when est_annonce_canonique then 1 end) as annonces_distinctes,
        count(distinct rome_code) as codes_rome,
        min(date_creation)::date as plus_ancienne,
        max(date_creation)::date as plus_recente
    from fct_offre
""").df()

,offres,annonces_distinctes,codes_rome,plus_ancienne,plus_recente
0,960,865,48,2025-05-21,2026-08-30


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : deux comptes, deux questions</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Le corpus compte <strong>960 offres</strong> pour <strong>865 annonces distinctes</strong>. L'écart vient des campagnes : un même poste publié dans plusieurs villes reçoit un identifiant par ville. Le dédoublonnage de <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">stg_ft_offres</code> travaille sur <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">offre_id</code> et écarte les doublons d'index de l'API, pas ces campagnes-là.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">« Combien d'offres » et « combien d'annonces » sont deux questions légitimes qui n'ont pas la même réponse. Le reste du notebook précise systématiquement laquelle il pose.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;"><span style="color:#FFC24B;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;">Point de vigilance :</span> <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">date_creation</code> s'étale sur 466 jours, ce qui pourrait laisser croire à un historique long. La section 1.3 montre que 90 % du corpus tient en trois mois.</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">1.2</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Nature des offres : contrat, expérience, métier</h3>
</div>

In [4]:
con.execute("""
    select
        type_contrat,
        count(*) as nb_offres,
        round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
    from fct_offre
    group by type_contrat
    order by nb_offres desc
""").df()

,type_contrat,nb_offres,pct
0,CDI,710,74.0
1,CDD,154,16.0
2,MIS,80,8.3
3,LIB,14,1.5
4,CCE,2,0.2


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : le marché data est un marché de CDI</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><strong>710 offres en CDI, soit 74,0 %</strong>, devant le CDD (154, 16,0 %) et la mission d'intérim (80, 8,3 %). Les professions libérales (<code style="background:#1E2128;padding:2px 6px;border-radius:4px;">LIB</code>, 14) et le contrat de création d'entreprise (<code style="background:#1E2128;padding:2px 6px;border-radius:4px;">CCE</code>, 2) sont résiduels.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">Ce résultat contredit l'image d'un marché tech dominé par le freelance. La nuance à garder : France Travail collecte mal les missions de freelance, qui passent surtout par des plateformes spécialisées et sortent donc de ce périmètre. Le 74 % mesure la part du CDI parmi les offres publiées ici, pas la part du CDI dans l'emploi data au sens large.</p>
</div>

In [5]:
con.execute("""
    select
        experience_exige,
        count(*) as nb_offres,
        round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
    from fct_offre
    group by experience_exige
    order by nb_offres desc
""").df()

,experience_exige,nb_offres,pct
0,E,540,56.3
1,D,420,43.8


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : la nomenclature d'expérience est binaire dans les faits</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Deux valeurs seulement : <strong>E</strong> (expérience exigée) sur 540 offres, <strong>D</strong> (débutant accepté) sur 420. La nomenclature France Travail prévoit aussi <strong>S</strong> (expérience souhaitée), <strong>absente du corpus</strong>.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">Ce sont des codes bruts, jamais traduits en base : la traduction en libellés lisibles vit dans la couche de présentation, et les modèles conservent les valeurs canoniques.</p>
</div>

In [6]:
con.execute("""
    select
        r.rome_libelle,
        count(distinct o.offre_id) as nb_offres,
        round(100.0 * count(distinct o.offre_id) / sum(count(distinct o.offre_id)) over (), 1) as pct
    from fct_offre o
    join dim_rome r using (rome_code)
    group by r.rome_libelle
    order by nb_offres desc
    limit 12
""").df()

,rome_libelle,nb_offres,pct
0,Data engineer,299,31.1
1,Data analyst,208,21.7
2,Data scientist,204,21.3
3,Analyste décisionnel - Business Intelligence,58,6.0
4,Architecte base de données,48,5.0
5,Développeur / Développeuse décisionnel - Busin...,37,3.9
6,Consultant décisionnel / Consultante décisionn...,24,2.5
7,Analyste de la performance sportive,8,0.8
8,Yield manager,8,0.8
9,Consultant fonctionnel / Consultante fonctionn...,6,0.6


<div style="background:#15171C;border:1.5px solid #FF6B5A;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#FF6B5A;margin:0 0 10px;">Limite : le tag ROME n'est pas fiable, et c'est mesurable</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Les trois premiers libellés (<strong>data engineer (299), data analyst (208), data scientist (204)</strong>) concentrent 711 offres sur 960, soit 74 %. Jusque-là, cohérent.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">La queue de distribution l'est beaucoup moins. Le corpus porte <strong>48 libellés ROME distincts</strong>, dont une trentaine à une ou deux offres : « Assistant comptable », « Documentaliste », « Attaché commercial », « Conseiller en gestion de carrière ». Ces offres sont entrées par les mots-clés ou par un tag ROME erroné de la source.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">Elles ne sont pas filtrées, et c'est une décision assumée depuis la Phase 1 : les retirer demanderait une règle construite sur peu d'exemples, exactement ce que ce projet s'interdit. Elles restent donc visibles et mesurables plutôt que masquées. Le bruit se chiffre : <strong>33 libellés portent une ou deux offres</strong>, pour 39 offres au total, soit 4,1 % du corpus.</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">1.3</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Temporalité : ce que le corpus couvre vraiment</h3>
</div>

In [7]:
con.execute("""
    select
        date_trunc('month', date_creation)::date as mois,
        count(*) as nb_offres,
        round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
    from fct_offre
    group by mois
    order by mois
""").df()

,mois,nb_offres,pct
0,2025-05-01,1,0.1
1,2025-06-01,2,0.2
2,2025-07-01,1,0.1
3,2026-01-01,2,0.2
4,2026-02-01,8,0.8
5,2026-03-01,7,0.7
6,2026-04-01,32,3.3
7,2026-05-01,40,4.2
8,2026-06-01,171,17.8
9,2026-07-01,354,36.9


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : 466 jours d'amplitude, trois mois de matière</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><code style="background:#1E2128;padding:2px 6px;border-radius:4px;">date_creation</code> va du 21/05/2025 au 30/08/2026, soit 466 jours. Le chiffre invite à parler d'historique. La distribution mensuelle dit autre chose : <strong>867 offres sur 960, soit 90,3 %, sont publiées en juin, juillet et août 2026</strong>. Les mois de 2025 portent quatre offres à eux tous.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Ces quelques offres anciennes ne sont pas des erreurs : ce sont des annonces republiées ou jamais dépubliées, dont la date de création remonte. Elles n'invalident rien, mais elles interdisent de lire ce corpus comme une série temporelle. Toute analyse d'évolution passe par les tables de flux hebdomadaires, construites pour ça (section 06).</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;"><span style="color:#FFC24B;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;">Point de vigilance :</span> un mois d'août à 342 offres pour un mois de juillet à 354 ne signifie pas que le marché a ralenti : la collecte s'arrête au 30/08, et un mois incomplet n'est pas comparable à un mois plein.</p>
</div>

<div style="background-color:#15171C;border-bottom:1px solid #2A2E37;padding:28px 32px;background-image: linear-gradient(rgba(255,255,255,.035) 1px, transparent 1px), linear-gradient(90deg, rgba(255,255,255,.035) 1px, transparent 1px); background-size: 24px 24px; max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 8px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">02. ÉTAPE</p>
<h2 style="margin:0;padding:0;letter-spacing:-0.02em;color:#F2F3F5;">Ce que le corpus n'est pas</h2>
</div>

<div style="padding:8px 0 24px 0;max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 8px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">Avertissement</p>
<h2 style="margin:0 0 16px 0;padding:0;letter-spacing:-0.02em;color:#F2F3F5;">Trois limites de comptage, toutes mesurées</h2>
<p style="color:#F2F3F5;line-height:1.65;">Cette étape n'apporte aucun résultat sur le marché. Elle établit trois choses que le corpus n'est pas, parce que chacune fausse une conclusion différente, et qu'aucune ne se voit sans la chercher.</p>
<p style="color:#F2F3F5;line-height:1.65;">Ces trois limites produisent chacune une erreur différente si on les ignore. Elles sont mesurées ici, puis neutralisées par une colonne dédiée dans le modèle, jamais par une suppression de données.</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif; max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">2.1</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Le corpus cumule, le marché renouvelle</h3>
</div>

In [8]:
con.execute("""
    select
        semaine,
        nb_actives,
        nb_nouvelles,
        nb_sorties,
        nb_reapparues,
        taux_sortie_pct,
        semaines_depuis_precedente
    from fct_marche_flux
    order by semaine
""").df()

,semaine,nb_actives,nb_nouvelles,nb_sorties,nb_reapparues,taux_sortie_pct,semaines_depuis_precedente
0,2026-07-13,552,552,<NA>,<NA>,NaN,<NA>
1,2026-08-24,497,408,463,0,83.9,6
2,2026-08-31,494,20,26,3,5.2,1


<div style="background:#15171C;border:1.5px solid #FF6B5A;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#FF6B5A;margin:0 0 10px;">Constat : fct_offre ne décroît jamais</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">La source <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">raw</code> unionne tous les dumps collectés et les dédoublonne. C'est le bon comportement pour un corpus, et le mauvais pour une mesure de marché : une offre vue une fois y reste pour toujours.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><strong>Sur les 552 offres de juillet, 463 avaient disparu de France Travail six semaines plus tard, soit 83,9 %</strong>, et <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">fct_offre</code> les comptait encore. Une courbe du nombre d'offres tirée de cette table raconte la taille du fichier, pas l'état du marché.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">D'où deux tables de faits distinctes : <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">fct_marche_hebdo</code> mesure le corpus accumulé, <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">fct_marche_flux</code> mesure la présence réelle des offres dans chaque collecte. Seule la seconde sait dire ce qui apparaît et ce qui disparaît.</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">2.2</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Une même annonce, plusieurs identifiants</h3>
</div>

In [9]:
con.execute("""
    select
        taille_grappe,
        count(distinct signature_annonce) as nb_grappes,
        count(*) as nb_offres
    from fct_offre
    where taille_grappe > 1
    group by taille_grappe
    order by taille_grappe desc
""").df()

,taille_grappe,nb_grappes,nb_offres
0,25,1,25
1,9,1,9
2,4,1,4
3,3,6,18
4,2,48,96


In [10]:
# La plus grosse grappe : un même texte, publié dans combien de communes ?
con.execute("""
    select
        min(intitule) as intitule,
        min(coalesce(entreprise_nom, '(non nommé)')) as employeur,
        count(*) as nb_offres,
        count(distinct cle_commune) as nb_communes,
        min(date_creation)::date as premiere,
        max(date_creation)::date as derniere
    from fct_offre
    group by signature_annonce
    having count(*) >= 4
    order by nb_offres desc
""").df()

,intitule,employeur,nb_offres,nb_communes,premiere,derniere
0,Data Governance Manager en Secteur Bancaire (H/F),LEIHIA,25,24,2026-07-11,2026-08-30
1,Data analyst (H/F),(non nommé),9,9,2026-08-15,2026-08-20
2,Hardware Verification Engineer H/F (H/F),(non nommé),4,2,2026-06-25,2026-06-26


<div style="background:#15171C;border:1.5px solid #FF6B5A;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#FF6B5A;margin:0 0 10px;">Constat : 15,8 % du corpus est une republication</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><strong>152 offres sur 960 partagent leur texte avec au moins une autre</strong>, réparties en 57 grappes. La plus grosse compte 25 offres : un employeur publiant le même poste dans <strong>24 communes</strong>, du 11 juillet au 30 août.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">La signature est l'empreinte du texte normalisé en minuscules et espaces réduits. Aucun seuil de similarité : deux textes sont identiques ou ils ne le sont pas. Une mesure de similarité attraperait davantage (sur onze annonces d'une même campagne outre-mer, neuf partagent exactement le même texte et deux en ont un légèrement différent), mais au prix d'un seuil que rien ne permet aujourd'hui de défendre.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Le risque de fausse grappe est écarté par la mesure : la description la plus courte du corpus fait 296 caractères, et 17 seulement passent sous 500. Aucune chance qu'un texte générique trop bref regroupe des offres sans rapport.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;"><code style="background:#1E2128;padding:2px 6px;border-radius:4px;">est_annonce_canonique</code> marque une offre par grappe, la plus ancienne. Filtrer dessus compte des annonces (865), ne pas filtrer compte des offres (960).</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">2.3</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Des salaires annuels qui n'en sont pas</h3>
</div>

In [11]:
con.execute("""
    select
        offre_id,
        salaire_min,
        categorie_employeur,
        cle_commune,
        intitule
    from fct_offre
    where salaire_annuel_plausible = false
    order by salaire_min, offre_id
""").df()

,offre_id,salaire_min,categorie_employeur,cle_commune,intitule
0,4933945,15,INTERMEDIAIRE,64240,Développeur BI (H/F)
1,5227728,40,EMPLOYEUR_DIRECT,44230,Data scientist (H/F)
2,5227732,40,EMPLOYEUR_DIRECT,49440,Data scientist (H/F)
3,6117320,40,EMPLOYEUR_DIRECT,31770,Chef de projet Data / Business Analyst (H/F)
4,5972977,1800,ANONYME,97233,Data analyst (H/F)
5,6003131,1800,ANONYME,97330,Data analyst (H/F)
6,6003150,1800,ANONYME,97380,Data analyst (H/F)
7,6029728,1800,ANONYME,97400,Data analyst (H/F)
8,6029796,1800,ANONYME,97411,Data analyst (H/F)
9,6039833,1800,ANONYME,97413,Data analyst (H/F)


<div style="background:#15171C;border:1.5px solid #FF6B5A;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#FF6B5A;margin:0 0 10px;">Constat : invisible sur l'agrégat, destructeur sur la tranche</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><strong>15 offres sur 275 portent un salaire annuel implausible</strong> : onze à 1 800 €, quatre entre 15 et 40 €. Deux mécanismes, tous deux des erreurs d'étiquetage de période à la source : un salaire mensuel et un taux horaire déclarés annuels.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Sur la médiane globale, elles ne changent <strong>rien</strong> : 45 000 € avec ou sans elles, parce que quinze valeurs sur 275 ne déplacent pas une médiane. Sur une tranche, elles la retournent. La médiane des offres mentionnant Tableau affichait <strong>1 800 € au lieu de 37 000</strong>, celle d'Excel 1 800 au lieu de 35 000. Ces deux outils sont associés aux profils juniors, donc ils concentraient les annonces au salaire mensuel mal étiqueté.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Pire : les onze annonces à 1 800 € sont toutes classées <strong>ANONYME</strong>, et à elles seules elles créaient un écart salarial apparent de 5 000 € entre employeurs masqués et employeurs directs. Une fois écartées, les trois catégories tombent exactement sur la même médiane.</p>
</div>

<div style="background-color:#15171C;border-bottom:1px solid #2A2E37;padding:28px 32px;background-image: linear-gradient(rgba(255,255,255,.035) 1px, transparent 1px), linear-gradient(90deg, rgba(255,255,255,.035) 1px, transparent 1px); background-size: 24px 24px;max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 8px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">03. ÉTAPE</p>
<h2 style="margin:0;padding:0;letter-spacing:-0.02em;color:#F2F3F5;">Compétences demandées</h2>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">3.1</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Technologies les plus citées</h3>
</div>

In [12]:
# Par ANNONCE : un employeur qui publie 25 fois le même texte ne crée pas
# 25 fois de la demande. La comparaison des deux comptages est en 3.2.
con.execute("""
    select
        t.technologie,
        count(distinct t.offre_id) as nb_annonces,
        round(100.0 * count(distinct t.offre_id)
              / (select count(*) from fct_offre where est_annonce_canonique), 1) as pct_annonces
    from fct_offre_technologie t
    join fct_offre o using (offre_id)
    where o.est_annonce_canonique
    group by t.technologie
    order by nb_annonces desc
    limit 12
""").df()

,technologie,nb_annonces,pct_annonces
0,Python,260,30.1
1,SQL,234,27.1
2,Power BI,170,19.7
3,AWS,57,6.6
4,Databricks,53,6.1
5,Azure,51,5.9
6,Tableau,51,5.9
7,Snowflake,44,5.1
8,Git,43,5.0
9,GCP,43,5.0


In [13]:
# L'écart entre les deux comptages, technologie par technologie.
con.execute("""
    select
        t.technologie,
        count(distinct t.offre_id) as par_offre,
        count(distinct case when o.est_annonce_canonique then t.offre_id end) as par_annonce,
        count(distinct t.offre_id)
          - count(distinct case when o.est_annonce_canonique then t.offre_id end) as ecart
    from fct_offre_technologie t
    join fct_offre o using (offre_id)
    group by t.technologie
    order by ecart desc
    limit 10
""").df()

,technologie,par_offre,par_annonce,ecart
0,SQL,282,234,48
1,Python,283,260,23
2,Power BI,188,170,18
3,Tableau,63,51,12
4,Excel,47,35,12
5,BigQuery,31,26,5
6,Snowflake,48,44,4
7,R,35,31,4
8,Databricks,57,53,4
9,C++,4,1,3


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : le dédoublonnage change le classement de tête</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Comptées par offre, Python (283) et SQL (282) semblent au coude-à-coude. Comptées par annonce, <strong>Python (262) passe devant SQL (235) de 11,5 %</strong>.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">SQL est la technologie la plus gonflée par les campagnes, et ça se comprend : c'est la compétence la plus générique, donc la plus présente dans les textes standardisés qu'on republie tels quels. Tableau perd 17,5 % de ses mentions pour la même raison.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">Le classement par offre n'était pas faux, il répondait à une autre question. Mais « quelle compétence le marché demande-t-il » se mesure en annonces, pas en publications.</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">3.2</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Le stack diffère-t-il selon l'expérience exigée ?</h3>
</div>

In [14]:
con.execute("""
    with base as (
        select offre_id, experience_exige
        from fct_offre
        where est_annonce_canonique
    ),
    total as (
        select experience_exige, count(*) as n from base group by experience_exige
    )
    select
        t.technologie,
        b.experience_exige,
        count(distinct t.offre_id) as nb_annonces,
        round(100.0 * count(distinct t.offre_id) / max(total.n), 1) as pct_du_niveau
    from fct_offre_technologie t
    join base b using (offre_id)
    join total on total.experience_exige = b.experience_exige
    where t.technologie = 'Python' or t.technologie = 'SQL' or t.technologie = 'Power BI'
    group by t.technologie, b.experience_exige
    order by t.technologie, b.experience_exige
""").df()

,technologie,experience_exige,nb_annonces,pct_du_niveau
0,Power BI,D,76,19.9
1,Power BI,E,94,19.5
2,Python,D,95,24.9
3,Python,E,165,34.2
4,SQL,D,75,19.6
5,SQL,E,159,32.9


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : Power BI est la seule technologie qui ne discrimine pas</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><strong>Python</strong> passe de 24,9 % des annonces débutant à <strong>34,2 %</strong> des annonces à expérience exigée, soit +9,3 points. <strong>SQL</strong> suit la même pente, 19,6 % contre 32,9 %, soit +13,3 points. <strong>Power BI</strong> reste plat : 19,9 % contre 19,5 %.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Lecture : Python et SQL sont des marqueurs de séniorité sur ce marché, Power BI est un outil demandé indifféremment à tous les niveaux. Pour une reconversion, ça dessine deux stratégies distinctes : Power BI ouvre des portes immédiatement, Python et SQL conditionnent la suite.</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">3.3</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Domaines d'intervention</h3>
</div>

In [15]:
con.execute("""
    select
        d.domaine_normalise,
        count(distinct case when o.est_annonce_canonique then d.offre_id end) as par_annonce,
        count(distinct d.offre_id) as par_offre
    from fct_offre_domaine d
    join fct_offre o using (offre_id)
    where d.domaine_normalise in (select distinct domaine_canonique from mapping_domaines)
    group by d.domaine_normalise
    order by par_annonce desc
""").df()

,domaine_normalise,par_annonce,par_offre
0,Analyse de données,175,192
1,Gouvernance des données,161,196
2,Business Intelligence,116,125
3,Machine Learning,103,107
4,Data Science,93,97
5,Qualité des données,86,111
6,Data Engineering,82,88
7,Gestion de projet,73,79
8,Architecture data,57,60
9,Deep Learning,43,43


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : deuxième classement retourné par le dédoublonnage</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Par offre, <strong>Gouvernance des données (196)</strong> devance <strong>Analyse de données (192)</strong>. Par annonce, l'ordre s'inverse : <strong>Analyse de données (175)</strong> devant <strong>Gouvernance (161)</strong>.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">La cause est identifiable à l'offre près : la plus grosse campagne du corpus est un poste de <em>Data Governance Manager</em> publié 25 fois. Un seul employeur suffisait à placer la gouvernance en tête d'un classement portant sur 960 offres.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">Rappel de portée : ce classement ne couvre que les domaines mappés dans <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">mapping_domaines</code>, soit <strong>19,7 % des mentions</strong>. La longue traîne n'est volontairement pas normalisée. Ce taux est identique à celui mesuré sur 552 offres, ce qui indique que la traîne grossit au même rythme que les douze clusters de tête : le mapping ne se dilue pas.</p>
</div>

<div style="background-color:#15171C;border-bottom:1px solid #2A2E37;padding:28px 32px;background-image: linear-gradient(rgba(255,255,255,.035) 1px, transparent 1px), linear-gradient(90deg, rgba(255,255,255,.035) 1px, transparent 1px); background-size: 24px 24px;max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 8px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">04. ÉTAPE</p>
<h2 style="margin:0;padding:0;letter-spacing:-0.02em;color:#F2F3F5;">Rémunération</h2>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">4.1</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Distribution des salaires annuels</h3>
</div>

In [16]:
# Deux filtres, deux raisons distinctes : salaire_annuel_plausible écarte les
# périodes mal étiquetées (2.3), est_annonce_canonique évite qu'un annonceur
# pèse autant de fois qu'il a publié (2.2).
con.execute("""
    select
        count(*) as nb_annonces,
        min(salaire_min) as minimum,
        quantile_cont(salaire_min, 0.25) as q1,
        median(salaire_min) as mediane,
        round(avg(salaire_min)) as moyenne,
        quantile_cont(salaire_min, 0.75) as q3,
        max(salaire_min) as maximum
    from fct_offre
    where salaire_periode = 'annuel'
      and salaire_annuel_plausible
      and est_annonce_canonique
""").df()

,nb_annonces,minimum,q1,mediane,moyenne,q3,maximum
0,216,25000,36000.0,43000.0,45355.0,50000.0,100000


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : 43 000 € de médiane, sur un quart du corpus</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><strong>216 annonces</strong> portent un salaire annuel exploitable. Médiane <strong>43 000 €</strong>, quartiles à 36 000 et 50 000 €, amplitude de 25 000 à 100 000 €.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">La moyenne (45 355 €) dépasse la médiane de 2 355 € : la distribution est étirée vers le haut par quelques postes seniors, ce qui est le profil attendu d'une distribution salariale et confirme que la médiane est le bon indicateur ici.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;"><span style="color:#FFC24B;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;">Point de vigilance :</span> 216 annonces sur 865, soit un quart. Et ce quart n'a aucune raison d'être représentatif : afficher son salaire est en soi un comportement d'employeur, mesuré en 4.3. Toute conclusion salariale porte donc sur les employeurs qui acceptent d'afficher, pas sur le marché.</p>
</div>

In [17]:
con.execute("""
    select
        round(salaire_min / 5000) * 5000 as palier,
        count(*) as nb_annonces
    from fct_offre
    where salaire_periode = 'annuel' and salaire_annuel_plausible and est_annonce_canonique
    group by palier
    order by palier
""").df()

,palier,nb_annonces
0,25000.0,3
1,30000.0,22
2,35000.0,39
3,40000.0,42
4,45000.0,26
5,50000.0,34
6,55000.0,10
7,60000.0,19
8,65000.0,9
9,70000.0,8


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : les salaires sont annoncés par paliers</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><strong>68,1 % des montants annuels sont des multiples de 5 000 €</strong>, et le seul palier de 50 000 € concentre 13,4 % des annonces chiffrées.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">Conséquence directe sur la lecture : aucune précision n'est revendiquable sous le palier de 5 000 €. Un écart de 2 000 € entre deux médianes ne veut rien dire ; un écart de 9 000 €, comme celui de la section 4.2, en veut un.</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">4.2</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Salaire par catégorie d'employeur et par expérience</h3>
</div>

In [18]:
con.execute("""
    select
        categorie_employeur,
        count(*) as nb_annonces,
        median(salaire_min) as salaire_median
    from fct_offre
    where salaire_periode = 'annuel' and salaire_annuel_plausible and est_annonce_canonique
    group by categorie_employeur
    order by salaire_median desc
""").df()

,categorie_employeur,nb_annonces,salaire_median
0,INTERMEDIAIRE_reclasse,3,65000.0
1,ANONYME,22,45000.0
2,INTERMEDIAIRE,85,45000.0
3,EMPLOYEUR_DIRECT,106,42000.0


<div style="background:#15171C;border:1.5px solid #FF6B5A;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#FF6B5A;margin:0 0 10px;">Constat : aucun écart lisible entre catégories d'employeur</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Employeur direct 42 000 €, intermédiaire 45 000 €, employeur masqué 45 000 €. Les trois tiennent <strong>dans un seul palier d'annonce</strong>, celui de 5 000 € établi en 4.1 : aucun écart n'est lisible ici, et l'ordre entre eux ne veut rien dire.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Avant nettoyage, l'employeur masqué affichait 40 000 € contre 45 000 € aux deux autres, et l'écart semblait réel. Il était entièrement produit par <strong>onze annonces d'un seul annonceur</strong> au salaire mensuel étiqueté annuel, toutes classées ANONYME. Un résultat qui aurait été publié tel quel sans le drapeau de plausibilité.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><code style="background:#1E2128;padding:2px 6px;border-radius:4px;">INTERMEDIAIRE_reclasse</code> affiche une médiane plus haute, mais sur un effectif à un chiffre. Le chiffre est affiché ici parce qu'un notebook d'exploration montre tout ; il est <strong>écarté du rapport</strong>, où toute catégorie sous dix annonces sort du graphique et est nommée en note.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">Ce qui sépare réellement les catégories n'est pas le montant, c'est le fait de l'afficher. Voir 4.3.</p>
</div>

In [19]:
con.execute("""
    select
        experience_exige,
        count(*) as nb_annonces,
        median(salaire_min) as salaire_median
    from fct_offre
    where salaire_periode = 'annuel' and salaire_annuel_plausible and est_annonce_canonique
    group by experience_exige
    order by salaire_median
""").df()

,experience_exige,nb_annonces,salaire_median
0,D,48,39000.0
1,E,168,45000.0


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : l'expérience exigée vaut 6 000 € de médiane</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">39 000 € contre 45 000 €, sur 48 et 168 annonces. L'écart de <strong>6 000 €</strong> dépasse le palier d'annonce de 5 000 €, il est donc lisible malgré la granularité grossière des montants, mais de peu, et il faudra le revérifier quand l'échantillon débutant aura dépassé la cinquantaine.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">C'est, avec la transparence salariale, le seul écart net que ce corpus produise sur la rémunération. Ni la catégorie d'employeur, ni la commune, ni le secteur d'activité ne discriminent aussi clairement.</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">4.3</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Qui affiche son salaire ?</h3>
</div>

In [20]:
con.execute("""
    select
        categorie_employeur,
        count(*) as nb_annonces,
        count(case when salaire_mentionne then 1 end) as avec_salaire,
        round(100.0 * count(case when salaire_mentionne then 1 end) / count(*), 1) as taux_pct
    from fct_offre
    where est_annonce_canonique
    group by categorie_employeur
    order by taux_pct desc
""").df()

,categorie_employeur,nb_annonces,avec_salaire,taux_pct
0,INTERMEDIAIRE,176,93,52.8
1,EMPLOYEUR_DIRECT,363,134,36.9
2,INTERMEDIAIRE_reclasse,29,4,13.8
3,ANONYME,297,27,9.1


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : le meilleur résultat du corpus</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><strong>52,8 % chez les intermédiaires nommés contre 9,1 % chez les employeurs masqués, soit près de six fois plus.</strong> 176 et 297 annonces : les effectifs sont solides des deux côtés, contrairement à la plupart des croisements salariaux de ce notebook.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Plus intéressant encore : les offres <strong>reclassées</strong> (celles que l'extraction LLM a identifiées comme des intermédiaires masquant leur client, via <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">client_final_masque</code>) se comportent comme les <strong>anonymes</strong>, pas comme les intermédiaires nommés.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">Elles affichent un salaire dans 13,8 % des cas, à comparer aux 9,1 % des anonymes et aux 52,8 % des intermédiaires nommés : elles se rangent du côté des premières, très loin des secondes. C'est une validation croisée de la reclassification, obtenue sans l'avoir cherchée : ces offres étaient classées ANONYME pour une raison, et masquer son client va avec masquer son salaire. Deux dimensions indépendantes du modèle qui se confirment l'une l'autre.</p>
</div>

<div style="background-color:#15171C;border-bottom:1px solid #2A2E37;padding:28px 32px;background-image: linear-gradient(rgba(255,255,255,.035) 1px, transparent 1px), linear-gradient(90deg, rgba(255,255,255,.035) 1px, transparent 1px); background-size: 24px 24px;max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 8px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">05. ÉTAPE</p>
<h2 style="margin:0;padding:0;letter-spacing:-0.02em;color:#F2F3F5;">Géographie</h2>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">5.1</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">La clé géographique n'est pas le code postal</h3>
</div>

In [21]:
con.execute("""
    select
        count(*) as offres,
        count(code_postal) as avec_code_postal,
        count(cle_commune) as avec_cle_unifiee,
        round(100.0 * count(code_postal) / count(*), 1) as pct_code_postal,
        round(100.0 * count(cle_commune) / count(*), 1) as pct_cle_unifiee
    from fct_offre
""").df()

,offres,avec_code_postal,avec_cle_unifiee,pct_code_postal,pct_cle_unifiee
0,960,764,859,79.6,89.5


In [22]:
# Les clés qui n'existent QUE sous forme de code INSEE.
con.execute("""
    select
        cle_commune,
        min(c.nom_commune) as commune,
        count(*) as nb_offres
    from fct_offre o
    join dim_commune c using (cle_commune)
    where o.code_postal is null
    group by cle_commune
    order by nb_offres desc
""").df()

,cle_commune,commune,nb_offres
0,75056,Paris,77
1,69123,Lyon,15
2,13055,Marseille,3


<div style="background:#15171C;border:1.5px solid #FF6B5A;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#FF6B5A;margin:0 0 10px;">Constat : Paris était sous-compté de moitié</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Paris, Lyon et Marseille sont les trois communes françaises à arrondissements. Elles n'ont pas de code postal unique, donc France Travail renvoie leur <strong>code INSEE de commune globale</strong> (75056, 69123, 13055) avec un code postal vide. Toutes les autres communes portent les deux.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Une dimension géographique indexée sur le seul code postal rate donc systématiquement les trois plus grandes villes du pays. <strong>95 offres concernées, dont 77 à Paris</strong> : le rapport affichait 74 offres parisiennes là où il y en a 151, et faisait passer Paris pour deux fois Lyon quand le rapport réel est de 3,3.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">La clé unifiée <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">coalesce(code_postal, commune)</code> porte la couverture de <strong>79,6 % à 89,5 %</strong> du corpus.</p>
</div>

In [23]:
con.execute("""
    select
        c.nom_commune,
        count(distinct o.offre_id) as nb_offres
    from fct_offre o
    join dim_commune c using (cle_commune)
    where c.nom_commune is not null and c.nom_commune != 'NON_RESOLU'
    group by c.nom_commune
    order by nb_offres desc
    limit 12
""").df()

,nom_commune,nb_offres
0,Paris,151
1,Lyon,46
2,Courbevoie,28
3,Nanterre,27
4,Toulouse,26
5,Nantes,24
6,Lille,21
7,Bordeaux,14
8,Puteaux,13
9,Boulogne-Billancourt,12


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Lecture : pourquoi la géographie se compte par offre</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">C'est la seule mesure du projet volontairement comptée par <strong>offre</strong> et non par annonce. Un poste ouvert dans vingt-quatre communes représente une opportunité dans chacune ; le compter une seule fois, dans la ville de la publication la plus ancienne, effacerait les vingt-trois autres.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">L'exception est signalée sous le graphique du rapport et dans son pied de page. Une exception silencieuse dans un livrable qui se veut auditable serait pire que pas d'exception du tout.</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">5.2</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Zones : métropole, outre-mer, inconnue</h3>
</div>

In [24]:
con.execute("""
    select
        zone_geographique,
        count(*) as nb_offres,
        round(100.0 * count(*) / sum(count(*)) over (), 1) as pct
    from fct_offre
    group by zone_geographique
    order by nb_offres desc
""").df()

,zone_geographique,nb_offres,pct
0,metropole,842,87.7
1,inconnue,101,10.5
2,outre-mer,17,1.8


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : restreindre à la métropole ne changerait rien</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><strong>Métropole 842, inconnue 101, outre-mer 17.</strong> La question « faut-il restreindre le périmètre à la France métropolitaine ? » a été tranchée par la mesure : exclure l'outre-mer déplace le taux d'employeur masqué de 0,6 point et laisse la médiane salariale identique, tout en perdant cinq employeurs réels et distincts.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">La zone est donc exposée comme <strong>dimension</strong> et non appliquée comme filtre : restreindre reste possible en une clause, à la demande, sans toucher à la spécification ni jeter de données.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;"><span style="color:#FFC24B;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;">Point de vigilance :</span> la vraie zone problématique n'est pas l'outre-mer, c'est l'inconnue : 101 offres, six fois plus, sans aucune information de lieu. Les compter comme métropole par défaut gonflerait celle-ci de 10 % du corpus sur une supposition, d'où une troisième valeur plutôt qu'un rattachement arbitraire.</p>
</div>

<div style="background-color:#15171C;border-bottom:1px solid #2A2E37;padding:28px 32px;background-image: linear-gradient(rgba(255,255,255,.035) 1px, transparent 1px), linear-gradient(90deg, rgba(255,255,255,.035) 1px, transparent 1px); background-size: 24px 24px;max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 8px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">06. ÉTAPE</p>
<h2 style="margin:0;padding:0;letter-spacing:-0.02em;color:#F2F3F5;">Flux du marché</h2>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">6.1</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Ce qui apparaît, ce qui disparaît</h3>
</div>

In [25]:
con.execute("""
    select
        semaine,
        semaines_depuis_precedente,
        nb_actives,
        nb_nouvelles,
        nb_sorties,
        nb_reapparues,
        taux_renouvellement_pct,
        taux_sortie_pct
    from fct_marche_flux
    order by semaine
""").df()

,semaine,semaines_depuis_precedente,nb_actives,nb_nouvelles,nb_sorties,nb_reapparues,taux_renouvellement_pct,taux_sortie_pct
0,2026-07-13,<NA>,552,552,<NA>,<NA>,100.0,NaN
1,2026-08-24,6,497,408,463,0,82.1,83.9
2,2026-08-31,1,494,20,26,3,4.0,5.2


<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Constat : 84 % de renouvellement en six semaines</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Trois points de mesure à ce jour. Entre la collecte du 13/07 et celle du 24/08, six semaines plus tard : <strong>552 offres actives, 463 disparues, 408 nouvelles, 497 actives à l'arrivée</strong>.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">L'identité de conservation tient à l'unité près : <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">552 − 463 + 408 + 0 = 497</code>. Elle relie quatre mesures calculées indépendamment : une agrégation, une première occurrence, deux anti-jointures symétriques. Si l'une dérive, l'égalité casse. C'est le test le plus fort du modèle.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;"><span style="color:#FFC24B;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;">Point de vigilance :</span> lire les taux avec <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">semaines_depuis_precedente</code>. Les deux premières collectes sont séparées de six semaines, pas d'une : un taux de sortie de 83,9 % couvre six semaines. La colonne est exposée plutôt que normalisée d'office, pour rendre impossible de lire ce chiffre comme un rythme hebdomadaire par inadvertance.</p>
</div>

<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Lecture : pourquoi le terme « réapparues » existe</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;"><code style="background:#1E2128;padding:2px 6px;border-radius:4px;">nb_nouvelles</code> compte les offres jamais vues auparavant. Une offre présente en juillet, absente en août, republiée ensuite n'est donc ni nouvelle ni survivante : elle entrerait dans les actives sans figurer au bilan.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">Le défaut n'était pas observable sur deux points de mesure, où toute offre absente du premier est forcément nouvelle. Il est apparu au troisième, et c'est le test de conservation qui l'a signalé en faisant échouer le pipeline. <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">nb_reapparues</code> ferme le bilan sans dénaturer <code style="background:#1E2128;padding:2px 6px;border-radius:4px;">nb_nouvelles</code>, qui garde son sens de marché : une offre réellement neuve.</p>
</div>

<div style="background-color:#15171C;border-bottom:1px solid #2A2E37;padding:28px 32px;background-image: linear-gradient(rgba(255,255,255,.035) 1px, transparent 1px), linear-gradient(90deg, rgba(255,255,255,.035) 1px, transparent 1px); background-size: 24px 24px;max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 8px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">07. ÉTAPE</p>
<h2 style="margin:0;padding:0;letter-spacing:-0.02em;color:#F2F3F5;">Visualisations</h2>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">7.1</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Technologies et rémunération</h3>
</div>

In [26]:
df_tech = con.execute("""
    select t.technologie, count(distinct t.offre_id) as nb_annonces
    from fct_offre_technologie t
    join fct_offre o using (offre_id)
    where o.est_annonce_canonique
    group by t.technologie
    order by nb_annonces desc
    limit 10
""").df().sort_values("nb_annonces")

fig = go.Figure(go.Bar(
    x=df_tech["nb_annonces"], y=df_tech["technologie"], orientation="h",
    text=df_tech["nb_annonces"], textposition="outside",
    marker_color="#4C8DFF", cliponaxis=False,
))
fig.update_layout(
    title="Dix technologies les plus demandées, par annonce distincte",
    height=420, xaxis=dict(showticklabels=False), yaxis=dict(gridcolor="rgba(0,0,0,0)"),
)
fig.show()

In [27]:
df_sal = con.execute("""
    select salaire_min
    from fct_offre
    where salaire_periode = 'annuel' and salaire_annuel_plausible and est_annonce_canonique
""").df()

fig = go.Figure(go.Histogram(x=df_sal["salaire_min"], nbinsx=20, marker_color="#4C8DFF"))
fig.add_vline(x=df_sal["salaire_min"].median(), line_color="#FFC24B", line_width=2,
              annotation_text="médiane", annotation_font_color="#FFC24B")
fig.update_layout(
    title="Distribution des salaires annuels annoncés (216 annonces)",
    xaxis_title="Salaire annuel minimum (€)", yaxis_title="Annonces", height=400,
)
fig.show()

<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Lecture : la forme confirme le choix de la médiane</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">L'histogramme montre les paliers de 5 000 € en peigne, et une queue étirée vers la droite. La moyenne est tirée par cette queue, la médiane non : c'est ce qui justifie de n'utiliser que la seconde dans tout le rapport.</p>
</div>

<div style="background:#15171C;border:1px solid #4C8DFF;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 6px;">7.2</p>
<h3 style="margin:0;padding:0;letter-spacing:-0.01em;color:#F2F3F5;">Séniorité et transparence</h3>
</div>

In [28]:
df_exp = con.execute("""
    with base as (select offre_id, experience_exige from fct_offre where est_annonce_canonique),
    total as (select experience_exige, count(*) as n from base group by experience_exige)
    select
        t.technologie,
        b.experience_exige,
        round(100.0 * count(distinct t.offre_id) / max(total.n), 1) as pct
    from fct_offre_technologie t
    join base b using (offre_id)
    join total on total.experience_exige = b.experience_exige
    where t.technologie = 'Python' or t.technologie = 'SQL' or t.technologie = 'Power BI'
    group by t.technologie, b.experience_exige
""").df()

libelles = {"D": "Débutant accepté", "E": "Expérience exigée"}
fig = go.Figure()
for niveau, couleur in [("D", "#4C8DFF"), ("E", "#FFC24B")]:
    part = df_exp[df_exp["experience_exige"] == niveau]
    fig.add_trace(go.Bar(
        x=part["technologie"], y=part["pct"], name=libelles[niveau],
        text=part["pct"].map(lambda v: f"{v} %"), textposition="outside",
        marker_color=couleur, cliponaxis=False,
    ))
fig.update_layout(
    title="Présence de la technologie selon l'expérience exigée",
    barmode="group", yaxis_title="% des annonces du niveau", height=400,
    legend=dict(orientation="h", y=1.08, x=0),
)
fig.show()

<div style="background:#15171C;border:1px solid #2A2E37;border-radius:16px;padding:20px 24px;font-family:Inter,sans-serif;max-width:10000px;box-sizing:border-box;">
<p style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;margin:0 0 10px;">Lecture : deux marqueurs de séniorité, un outil transverse</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0 0 12px;">Le graphique rend visible d'un coup d'œil ce que la section 3.2 mesure : deux barres qui s'écartent nettement pour Python et SQL, deux barres à la même hauteur pour Power BI.</p>
<p style="color:#F2F3F5;line-height:1.65;margin:0;">Pour une reconversion, c'est la lecture la plus actionnable de tout ce notebook : Power BI ouvre des portes au même taux quel que soit le niveau, Python et SQL conditionnent l'accès aux postes qui exigent de l'expérience.</p>
</div>

In [29]:
df_transp = con.execute("""
    select
        categorie_employeur,
        round(100.0 * count(case when salaire_mentionne then 1 end) / count(*), 1) as taux_pct,
        count(*) as nb_annonces
    from fct_offre
    where est_annonce_canonique
    group by categorie_employeur
    order by taux_pct
""").df()

libelles_cat = {
    "EMPLOYEUR_DIRECT": "Employeur direct",
    "INTERMEDIAIRE": "Intermédiaire",
    "INTERMEDIAIRE_reclasse": "Intermédiaire (reclassé)",
    "ANONYME": "Employeur masqué",
}
fig = go.Figure(go.Bar(
    x=df_transp["taux_pct"],
    y=df_transp["categorie_employeur"].map(libelles_cat),
    orientation="h",
    text=[f"{t} %  ·  n={n}" for t, n in zip(df_transp["taux_pct"], df_transp["nb_annonces"])],
    textposition="outside", marker_color="#4C8DFF", cliponaxis=False,
))
fig.update_layout(
    title="Part des annonces affichant un salaire, par catégorie d'employeur",
    height=360, xaxis=dict(showticklabels=False), yaxis=dict(gridcolor="rgba(0,0,0,0)"),
)
fig.show()

<div style="padding:8px 0 24px 0;max-width:10000px;box-sizing:border-box;">
<p style="margin:0 0 8px 0;font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:0.08em;text-transform:uppercase;color:#4C8DFF;">Synthèse</p>
<h2 style="margin:0 0 16px 0;padding:0;letter-spacing:-0.02em;color:#F2F3F5;">Ce que ce notebook établit</h2>
<p style="color:#F2F3F5;line-height:1.65;"><strong>Trois résultats tiennent.</strong> La transparence salariale sépare nettement les catégories d'employeur, près de six fois plus d'affichage chez les intermédiaires nommés que chez les employeurs masqués. L'expérience exigée vaut 6 000 € de médiane. Python et SQL sont des marqueurs de séniorité, Power BI n'en est pas un.</p>
<p style="color:#F2F3F5;line-height:1.65;"><strong>Un résultat s'est retourné en cours de route</strong>, et deux classements avec lui. Python n'est pas stable d'un niveau d'expérience à l'autre comme le concluait la version précédente de ce notebook ; il devance SQL une fois les campagnes neutralisées ; et l'Analyse de données passe devant la Gouvernance pour la même raison.</p>
<p style="color:#F2F3F5;line-height:1.65;"><strong>Ce que le corpus ne permet pas de dire.</strong> Aucun écart salarial lisible entre catégories d'employeur, toutes tenant dans un palier d'annonce. Rien sur le profil d'entreprise, faute d'un SIREN sur plus de 17,8 % des offres. Rien sur une évolution temporelle depuis fct_offre, dont 90 % du contenu tient en trois mois. Et rien de précis sous 5 000 €, granularité réelle des montants annoncés.</p>
<p style="color:#F2F3F5;line-height:1.65;">La suite est du ressort du pipeline, pas du notebook : chaque correction identifiée ici a été portée dans les modèles dbt, avec un test pour qu'elle ne se reperde pas.</p>
</div>

In [30]:
con.close()